In [1]:
import numpy as np
import pandas as pd

In [2]:
scotus = pd.read_csv("data/scotus.csv", dtype=str)
fed = pd.read_csv("data/fed.csv", dtype=str)
state = pd.read_csv("data/state.csv", dtype=str)

In [3]:
df = pd.concat([scotus, fed, state], ignore_index=True)

df.head()

,citing-cited,citing_id,cited_id,batch,round1_expert,round1_treatment,round1_note,round2_expert,round2_treatment,round2_note,round1_round2,round3_expert,round3_treatment,round3_note,recon_treatment,final_treatment
0,100397-100087,100397,100087,SCOTUS,Catherine McCarthy,Cited by,NaN,Ursula Gorham,Cited by,NaN,TRUE,NaN,NaN,NaN,Cited by,Cited by
1,100397-100149,100397,100149,SCOTUS,Catherine McCarthy,Cited by,NaN,Ursula Gorham,Cited by,NaN,TRUE,NaN,NaN,NaN,Cited by,Cited by
2,100397-100256,100397,100256,SCOTUS,Catherine McCarthy,Cited by,NaN,Ursula Gorham,Cited by,NaN,TRUE,NaN,NaN,NaN,Cited by,Cited by
3,100397-3643566,100397,3643566,SCOTUS,Catherine McCarthy,Cited by,NaN,Ursula Gorham,Cited by,NaN,TRUE,NaN,NaN,NaN,Cited by,Cited by
4,100397-85452,100397,85452,SCOTUS,Catherine McCarthy,Cited by,NaN,Ursula Gorham,Cited by,NaN,TRUE,NaN,NaN,NaN,Cited by,Cited by


In [4]:
len(df)

11683

In [5]:
df_valid = df[df["final_treatment"] != "PENDING"]
len(df_valid)

8928

In [6]:
df_valid = df_valid.drop_duplicates()
len(df_valid)

8908

In [7]:
df_valid["final_treatment"].value_counts()

final_treatment
Cited by                                 8340
Distinguished by                          242
Ambiguous: Stop                           111
Affirmed by                                51
Reversed by                                45
Criticized by                              23
Ambiguous: Caution                         19
Ambiguous: Warning                         18
Overruled as recognized by                 12
Overruled by                                8
Reversed and remanded by                    6
Vacated and remanded by                     5
Declined to follow by                       4
Questioned by                               3
Abrogated by                                3
Affirmed as recognized by                   3
Limited by                                  3
Dismissed by                                2
Disapproved by                              2
Affirmed in part; Reversed in part by       2
Distinguished as recognized by              2
Criticized as reco

In [8]:
# Define severity mapping
severity_mapping = {
    "Reversed by": "Stop",
    "Reversed and remanded by": "Stop",
    "Vacated and remanded by": "Stop",
    "Vacated by": "Stop",
    "Overruled by": "Stop",
    "Abrogated by": "Stop",
    "Questioned by": "Stop",
    "Affirmed in part; Reversed in part by": "Warning",
    "Disapproved by": "Warning",
    "Limited by": "Warning",
    "Remanded by": "Caution",
    "Criticized by": "Caution",
    "Distinguished by": "Caution",
    "Declined to follow by": "Caution",
    "Ambiguous: Stop": "Stop",
    "Ambiguous: Warning": "Warning",
    "Ambiguous: Caution": "Caution",
    "Dismissed by": "Neutral",
    "Affirmed by": "Neutral",
    "Cited by": "Neutral"
}

# Define direction mapping
direction_mapping = {
    "Reversed by": "Direct History",
    "Reversed and remanded by": "Direct History",
    "Vacated and remanded by": "Direct History",
    "Vacated by": "Direct History",
    "Overruled by": "Citing Reference",
    "Abrogated by": "Citing Reference",
    "Questioned by": "Citing Reference",
    "Affirmed in part; Reversed in part by": "Direct History",
    "Disapproved by": "Citing Reference",
    "Limited by": "Citing Reference",
    "Remanded by": "Direct History",
    "Criticized by": "Citing Reference",
    "Distinguished by": "Citing Reference",
    "Declined to follow by": "Citing Reference",
    "Ambiguous: Stop": "Ambiguous",
    "Ambiguous: Warning": "Ambiguous",
    "Ambiguous: Caution": "Ambiguous",
    "Dismissed by": "Direct History",
    "Affirmed by": "Direct History",
    "Cited by": "Citing Reference"
}

# Add severity column — for "as recognized by" treatments, look up the base treatment type
df_valid["severity"] = df_valid["final_treatment"].apply(
    lambda x: severity_mapping.get(x.replace(" as recognized", ""), None)
    if x.endswith("as recognized by")
    else severity_mapping.get(x, None)
)

# Add direction column with additional condition for "as recognized by"
df_valid["direction"] = df_valid["final_treatment"].apply(
    lambda x: "Related Reference" if "as recognized by" in x else direction_mapping.get(x, None)
)

# Display the updated dataframe
df_valid.head()

,citing-cited,citing_id,cited_id,batch,round1_expert,round1_treatment,round1_note,round2_expert,round2_treatment,round2_note,round1_round2,round3_expert,round3_treatment,round3_note,recon_treatment,final_treatment,severity,direction
0,100397-100087,100397,100087,SCOTUS,Catherine McCarthy,Cited by,NaN,Ursula Gorham,Cited by,NaN,TRUE,NaN,NaN,NaN,Cited by,Cited by,Neutral,Citing Reference
1,100397-100149,100397,100149,SCOTUS,Catherine McCarthy,Cited by,NaN,Ursula Gorham,Cited by,NaN,TRUE,NaN,NaN,NaN,Cited by,Cited by,Neutral,Citing Reference
2,100397-100256,100397,100256,SCOTUS,Catherine McCarthy,Cited by,NaN,Ursula Gorham,Cited by,NaN,TRUE,NaN,NaN,NaN,Cited by,Cited by,Neutral,Citing Reference
3,100397-3643566,100397,3643566,SCOTUS,Catherine McCarthy,Cited by,NaN,Ursula Gorham,Cited by,NaN,TRUE,NaN,NaN,NaN,Cited by,Cited by,Neutral,Citing Reference
4,100397-85452,100397,85452,SCOTUS,Catherine McCarthy,Cited by,NaN,Ursula Gorham,Cited by,NaN,TRUE,NaN,NaN,NaN,Cited by,Cited by,Neutral,Citing Reference


In [9]:
assert df_valid["severity"].isna().sum() == 0
assert df_valid["direction"].isna().sum() == 0

In [10]:
cols = ['citing-cited', 'citing_id', 'cited_id', 'batch', 'final_treatment', 'severity', 'direction']
df_valid = df_valid[cols]

df_valid.head()

,citing-cited,citing_id,cited_id,batch,final_treatment,severity,direction
0,100397-100087,100397,100087,SCOTUS,Cited by,Neutral,Citing Reference
1,100397-100149,100397,100149,SCOTUS,Cited by,Neutral,Citing Reference
2,100397-100256,100397,100256,SCOTUS,Cited by,Neutral,Citing Reference
3,100397-3643566,100397,3643566,SCOTUS,Cited by,Neutral,Citing Reference
4,100397-85452,100397,85452,SCOTUS,Cited by,Neutral,Citing Reference


In [11]:
df_valid["final_treatment"].value_counts()

final_treatment
Cited by                                 8340
Distinguished by                          242
Ambiguous: Stop                           111
Affirmed by                                51
Reversed by                                45
Criticized by                              23
Ambiguous: Caution                         19
Ambiguous: Warning                         18
Overruled as recognized by                 12
Overruled by                                8
Reversed and remanded by                    6
Vacated and remanded by                     5
Declined to follow by                       4
Questioned by                               3
Abrogated by                                3
Affirmed as recognized by                   3
Limited by                                  3
Dismissed by                                2
Disapproved by                              2
Affirmed in part; Reversed in part by       2
Distinguished as recognized by              2
Criticized as reco

In [12]:
df_valid["severity"].value_counts()

severity
Neutral    8396
Caution     291
Stop        196
Warning      25
Name: count, dtype: int64

In [13]:
df_valid["direction"].value_counts()

direction
Citing Reference     8628
Ambiguous             148
Direct History        112
Related Reference      20
Name: count, dtype: int64

In [14]:
df_valid["batch"].value_counts()

batch
SCOTUS     6451
FEDERAL    1623
STATE       834
Name: count, dtype: int64

In [15]:
df_valid.to_csv("data/labels.csv", index=False)